This notebook is used to load and test the HRS RESPONDENT table.  It extracts distinct HHIDPN values from the RAND longitudinal data and populate the table.

**Purpose:** Load the HRS RESONDENT reference table.

**Source Table:** `dev_catalog.brz_raw_hrs.randhrs1992_2022v1`  
**Target Table:** `dev_catalog.slv_cdm_hrs.resondent`
**Load Script:** `../../sql/dml/load_hrs_respondent_data.sql`
**Validation Script:** `../../sql/validataion/verify_hrs_respondent_data.sql`

**Process:**
1. Clear/truncate the HRS RESONDENT table .
2. Extract distinct HHIDPN values from SOURCE data and load the TARGET table.
3. Validate the table data.
4. Display summary stats.

In [ ]:
# -----------------------------------------------------------------------------
# Initialize Notebook Configuration
# -----------------------------------------------------------------------------
# For Asset Bundles
#   Instead of relying on relative paths, add the bundle root to Python's path.
#   In each notebook that imports src, add this before the import:
import sys
sys.path.append("/Workspace/Users/peteperez.lv@gmail.com/.bundle/hrs_dbx_repo/default/files")

dbutils.widgets.dropdown(
    "truncate_table",
    "true",
    ["true", "false"]
)

TRUNCATE_TABLE = dbutils.widgets.get("truncate_table").lower() == "true"

TARGET_TABLE = "dev_catalog.slv_cdm_hrs.hrs_respondent"

LOAD_SQL = "../../sql/dml/load_hrs_respondent_data.sql"

VALIDATION_SQL = "../../sql/validation/verify_hrs_respondent_data.sql"

SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 1:
# Clear existing cohort data if needed (use with caution)
# Uncomment the line below to truncate the table before loading 

if TRUNCATE_TABLE:
    print("======================================================")
    print("Step 1 - TRUNCATE")
    print("======================================================")

    try:
        spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        print("✓ Completed")
    except Exception as e:
        print(f"❌ TRUNCATE failed: {e}")
        raise

else:
    print("Table not found.  Skipping table truncation.")

In [ ]:


# importlib to eliminate cache issues.
import importlib
import src.common.sql_utils as sql_utils

importlib.reload(sql_utils)

from src.common.sql_utils import execute_sql_file

import sys
sys.path.append("/Workspace/Users/peteperez.lv@gmail.com/.bundle/hrs_dbx_repo/default/files")

print("======================================================")
print("Step 2 - LOAD DATA")
print("======================================================")
try: 
    execute_sql_file(
        spark, 
        LOAD_SQL
    )
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise



In [ ]:
# # Step 3: Verify the TARGET_TABLE.

print("======================================================")
print("Step 3 - Validation")
print("======================================================")
try:
    execute_sql_file(
        spark,
        VALIDATION_SQL,
        display_results=True
    )
    print("✓ Completed")
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

In [ ]:
# Step 4 - Display summary statistics

source_count = spark.sql("""
    SELECT COUNT(DISTINCT HHIDPN) as distinct_hhidpn
    FROM dev_catalog.brz_raw_hrs.randhrs1992_2022v1
    WHERE HHIDPN IS NOT NULL
""").collect()[0][0]

target_count = spark.sql("""
    SELECT COUNT(*) as resondent_count
    FROM dev_catalog.slv_cdm_hrs.hrs_respondent
""").collect()[0][0]

print("=" * 60)
print("HRS HHIDPN DATA LOAD SUMMARY")
print("=" * 60)
print(f"Distinct HHIDPN values in source: {source_count}")
print(f"Total records in HRS Respondent table:      {target_count}")
print("=" * 60)

if source_count == target_count:
    print("✓ SUCCESS: All distinct HRS HHIDPN loaded")
else:
    print(f"⚠ WARNING: Mismatch detected. Please review.")

: 